# Ollama 세팅

In [ ]:
# 1. zstd 설치 (에러 해결용)
!sudo apt-get update && sudo apt-get install -y zstd

# 2. Ollama 재설치 시도
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,383 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 https://cli.github.com/packages stable

# Ollma 서버 실행

In [ ]:
import subprocess
import time

# 1. Ollama 서버를 백그라운드에서 실행
# stdout/stderr를 무시하지 않고 연결해두면 나중에 로그 확인이 가능합니다.
process = subprocess.Popen(['ollama', 'serve'],
                           stdout=subprocess.PIPE,
                           stderr=subprocess.PIPE,
                           text=True)

# 2. 서버가 완전히 준비될 때까지 대기
print("Ollama 서버를 시작하는 중...")
time.sleep(10)
print("Ollama 서버가 준비되었습니다!")

Ollama 서버를 시작하는 중...
Ollama 서버가 준비되었습니다!


In [ ]:
# 'llama3' 모델 다운로드 (약 4.7GB)
!ollama pull llama3

In [ ]:
# 모델 목록 확인
!ollama list

NAME             ID              SIZE      MODIFIED               
llama3:latest    365c0bd3c000    4.7 GB    Less than a second ago    


#모델 테스트

In [ ]:
# 프로세스가 살아있는지 확인 (목록에 'ollama serve'가 보여야 합니다)
!ps aux | grep ollama

root        3600 46.1  0.3 2435536 44456 ?       Sl   04:26   0:28 ollama serve
root        3954  0.0  0.0   7376  3500 ?        S    04:27   0:00 /bin/bash -c ps aux | grep ollama
root        3956  0.0  0.0   6484  2384 ?        S    04:27   0:00 grep ollama


In [ ]:
import requests
import json

def test_ollama(prompt):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "llama3",
        "prompt": prompt,
        "stream": False # 결과를 한 번에 받기 위해 False 설정
    }

    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()
        result = response.json()
        return result['response']
    except Exception as e:
        return f"에러 발생: {e}"

# 테스트 질문 실행
question = "IPCC 6차 보고서의 핵심 주제 3가지만 짧게 알려줘."
answer = test_ollama(question)

print(f"질문: {question}")
print("-" * 30)
print(f"답변:\n{answer}")

질문: IPCC 6차 보고서의 핵심 주제 3가지만 짧게 알려줘.
------------------------------
답변:
The IPCC Sixth Assessment Report (AR6) has several key themes, but here are three of the most important ones:

1. **Climate Change Mitigation**: The report emphasizes the urgent need to reduce greenhouse gas emissions to limit global warming to 1.5°C above pre-industrial levels and avoid catastrophic climate change impacts.
2. **Climate Change Impacts and Vulnerability**: The report highlights the devastating effects of climate change, such as sea-level rise, extreme weather events, and water scarcity, and emphasizes the need for adaptation measures to protect vulnerable populations and ecosystems.
3. **Climate Justice and Equity**: The report underscores the importance of climate justice and equity, emphasizing that climate change disproportionately affects poor and marginalized communities, and that global cooperation and equitable solutions are essential to address this crisis.

These themes are interconnected an

In [ ]:
# NVIDIA GPU 사용 현황 확인
!nvidia-smi

Thu Feb 26 04:28:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             32W /   70W |    5119MiB /  15360MiB |     90%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#RAG 시스템 구축

In [ ]:

!pip install langchain-core
!pip install -U langchain langchain-community langchain-huggingface langchain-text-splitters pymupdf sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.0 

In [ ]:
import re
# 최신 버전의 경로입니다.
from langchain_core.documents import Document

def sliding_window_chunking(docs, window_size=5, step=2):
    """문장 단위로 슬라이딩 윈도우 청킹 수행 (JCCR 논문 전략 반영)"""
    chunked_docs = []

    for doc in docs:
        # 1. 문장 단위로 분리 (마침표, 물음표 등 기준)
        sentences = re.split(r'(?<=[.!?])\s+', doc.page_content)

        # 2. 슬라이딩 윈도우 적용
        for i in range(0, len(sentences), step):
            window = sentences[i : i + window_size]
            if not window: continue

            chunk_content = " ".join(window)

            # 메타데이터 유지 및 청크 정보 추가
            new_metadata = doc.metadata.copy()
            new_metadata["chunk_index"] = i // step

            chunked_docs.append(Document(page_content=chunk_content, metadata=new_metadata))

            # 마지막 문단 처리
            if i + window_size >= len(sentences): break

    return chunked_docs

print("✅ 청킹 함수가 정상적으로 정의되었습니다.")

✅ 청킹 함수가 정상적으로 정의되었습니다.


In [ ]:
# Mount Google Drive if not already mounted
from google.colab import drive
drive.mount('/content/drive')

import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Updated import for HuggingFaceEmbeddings as per LangChain deprecation warning
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


# 1. 설정 (이전 단계에서 했던 설정 유지)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# 영어 문서용 임베딩 모델로 변경
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("✅ RAG 구성 요소 및 임베딩 모델이 로드되었습니다.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ RAG 구성 요소 및 임베딩 모델이 로드되었습니다.


In [ ]:
# 파일 로드 및 청킹 통합 실행
all_processed_chunks = []

# Google Drive 내 IPCC_data 폴더 경로
BASE_PATH = "/content/drive/MyDrive/IPCC_data/"

# 파일 목록을 Google Drive의 실제 파일명으로 업데이트합니다.
# (사용자 제공 예시를 바탕으로 다른 파일명 추정)
ipcc_files = [
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Section_1.pdf", # '세션1.pdf'에 해당한다고 추정
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Section_2.pdf",    # '세션2.pdf'에 해당한다고 추정
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Section_3.pdf",    # '세션3.pdf'에 해당한다고 추정
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Section_4.pdf",    # '세션4.pdf'에 해당한다고 추정
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Annex_1_Glossary.pdf", # '부속서_용어집.pdf'에 해당 (사용자 제공)
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Annex_2_Acronyms.pdf"  # '부속서_약어.pdf'에 해당한다고 추정
]

for file_name in ipcc_files:
    if os.path.exists(file_name):
        loader = PyMuPDFLoader(file_name)
        pages = loader.load()

        # 각 파일의 메타데이터 주입
        # 파일명에서 특정 접두사와 .pdf 확장자를 제거하여 섹션 이름을 추출합니다.
        section_name = os.path.basename(file_name).replace("IPCC_AR6_SYR_FullVolume_", "").replace(".pdf", "")
        for p in pages: p.metadata["section"] = section_name

        # 논문 기반 슬라이딩 윈도우 청킹 적용
        chunks = sliding_window_chunking(pages)
        all_processed_chunks.extend(chunks)
        print(f"✅ {section_name}: {len(chunks)}개 청크 생성 완료")
    else:
        print(f"❌ 파일이 존재하지 않습니다: {file_name}. Google Drive에 파일이 올바르게 업로드되었는지 확인해주세요.")

# 벡터 DB 생성
if all_processed_chunks:
    vectorstore = Chroma.from_documents(documents=all_processed_chunks, embedding=embeddings)
    print("🔥 모든 파일이 벡터 데이터베이스에 저장되었습니다!")
else:
    print("⚠️ 처리된 청크가 없어 벡터 데이터베이스를 생성할 수 없습니다.")

✅ Section_1: 18개 청크 생성 완료
✅ Section_2: 250개 청크 생성 완료
✅ Section_3: 209개 청크 생성 완료
✅ Section_4: 246개 청크 생성 완료
✅ Annex_1_Glossary: 176개 청크 생성 완료
✅ Annex_2_Acronyms: 8개 청크 생성 완료
🔥 모든 파일이 벡터 데이터베이스에 저장되었습니다!


In [ ]:
# 테스트 실행
query = "What is the definition of Net Zero?"
test_results = vectorstore.similarity_search(query, k=3)

for doc in test_results:
    print(f"출처: {doc.metadata['section']}")
    print(f"내용: {doc.page_content[:150]}...")
    print("-" * 30)

출처: Annex_1_Glossary
내용: The quantification of net zero 
GHG emissions depends on the GHG emission metric chosen to compare 
emissions and removals of different gases, as well...
------------------------------
출처: Section_2
내용: Achieving net zero emission targets relies on policies, institutions, and milestones against which to track progress. Least-cost global 
modelled path...
------------------------------
출처: Section_2
내용: The potential and cost of achieving net zero or even net negative emissions 
vary by sector and region. If and when net zero emissions for a given sec...
------------------------------


# Ollama RAG 체인 완성

In [ ]:
def ipcc_ai_analyst(question):
    # 1. 지식 검색 (상위 4개 조각 추출)
    docs = vectorstore.similarity_search(question, k=4)

    # 2. 컨텍스트 구성 및 출처 정리
    context = "\n\n".join([f"[Source: {d.metadata['section']}]\n{d.page_content}" for d in docs])
    sources = list(set([d.metadata['section'] for d in docs]))

    # 3. IPCC 전용 프롬프트 설계 (JCCR 논문 기반 페르소나 적용)
    prompt = f"""
너는 IPCC 6차 보고서(AR6) 전문 분석가야. 아래 제공된 [Context]를 바탕으로 질문에 답해줘.

[Context]
{context}

[Question]
{question}

[Guidelines]
- 반드시 제공된 Context의 내용을 근거로 답변할 것.
- 답변은 전문적이고 신뢰감 있는 한국어로 작성할 것.
- 인류의 활동에 의한 기후 변화의 책임을 명확히 전달할 것 (Action-oriented).
- 전문 용어는 영어 원문을 괄호 안에 병기할 것 (예: 탄소 중립(Net Zero)).
- 답변 마지막에 참고한 섹션들을 나열할 것.

답변:
"""

    # 4. Ollama 호출 (우리가 이전에 만든 test_ollama 함수 사용)
    response = test_ollama(prompt)

    print(f"\n📢 분석 결과 (참조: {', '.join(sources)})")
    print("-" * 50)
    print(response)

# 🔥 드디어 실행!
ipcc_ai_analyst("탄소 중립(Net Zero)의 정의와 이를 달성하기 위해 필요한 정책적 노력은 무엇인가요?")


📢 분석 결과 (참조: Section_3, Annex_1_Glossary, Section_2)
--------------------------------------------------
**탄소 중립(Net Zero)**는 기후變化의 온실가스(Greenhouse Gas, GHG) 배출을 완전히 OFFSET하거나 제로하게 하는 상태를 의미합니다. 이를 달성하기 위해 필요한 정책적 노력은 다음과 같습니다.

첫째, **탄소 중립 추구** : 각국은 탄소 중립을 목표로 하고, 그에 적절한 계획과 조치를 세워야 합니다. 국제협정인 파리기후협정(PARIS Agreement)에서도 이와 같은 노력이 필요하다고 강조하고 있습니다.

둘째, **GHG 배출 감축** : 각국은 GHG 배출을 줄이기 위해 다양한 정책을 개발하여야 합니다. 예를 들어, 재생에너지의 확대, 저탄소 에너지 사용, 공유 교통의 개발 등이 필요합니다.

셋째, **탄소 저장** : 탄소 중립을 달성하기 위해는 또한 탄소 저장을 고려해야 합니다. 이를 통해 기후변화의 온실가스 배출을 완전히 OFFSET할 수 있습니다.

넷째, **정책적 지원** : 각국은 탄소 중립을 달성하기 위해 필요한 정책적 지원을 제공해야 합니다. 예를 들어, 인프라 개발, 기술 개발, 교육 등이 필요합니다.

이러한 정책적 노력들이 모여탄소 중립을 달성할 수 있습니다. **탄소 중립 추구**는 기후변화의 온실가스 배출을 완전히 OFFSET하거나 제로하게 하는 상태를 의미하며, 이를 달성하기 위해 필요한 정책적 노력이므로 중요한 주제입니다.

참고:
* Annex 1 Glossary
* Section 2
* Section 3
